# 14. Multi-Target UniProt Metadata

This notebook collects reliable biological metadata for each project target using UniProt.

Goal:
- Confirm each target has the correct protein/gene identity.
- Collect target function summaries.
- Collect standard external identifiers such as Ensembl, ChEMBL, HGNC, and PDB links.
- Save raw UniProt API responses and processed metadata tables.

This notebook is part of the data preparation stage. It helps make the final knowledge base more trustworthy.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "uniprot"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_FILE = PROCESSED_DIR / "multi_target_chembl_targets.csv"

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Raw folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/uniprot
Processed folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


## 1. Define Target Accessions

These are canonical human UniProt accessions for the project targets.

In [3]:
TARGET_UNIPROT_ACCESSIONS = {
    "EGFR": "P00533",
    "ERBB2": "P04626",
    "BRAF": "P15056",
    "ALK": "Q9UM73",
    "KRAS": "P01116",
    "VEGFA": "P15692",
    "MET": "P08581",
    "PIK3CA": "P42336",
}

TARGET_DISPLAY_NAMES = {
    "EGFR": "EGFR",
    "ERBB2": "HER2 / ERBB2",
    "BRAF": "BRAF",
    "ALK": "ALK",
    "KRAS": "KRAS",
    "VEGFA": "VEGFA",
    "MET": "MET",
    "PIK3CA": "PIK3CA",
}

print("Targets:", len(TARGET_UNIPROT_ACCESSIONS))
for target_symbol, accession in TARGET_UNIPROT_ACCESSIONS.items():
    print(f"{target_symbol}: {accession}")

Targets: 8
EGFR: P00533
ERBB2: P04626
BRAF: P15056
ALK: Q9UM73
KRAS: P01116
VEGFA: P15692
MET: P08581
PIK3CA: P42336


In [4]:
if TARGETS_FILE.exists():
    chembl_targets_df = pd.read_csv(TARGETS_FILE)
    print("Loaded ChEMBL target file:", TARGETS_FILE)
    display(chembl_targets_df)
else:
    chembl_targets_df = pd.DataFrame({"target_symbol": list(TARGET_UNIPROT_ACCESSIONS)})
    print("ChEMBL target file not found. Continuing with configured target list only.")

Loaded ChEMBL target file: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_chembl_targets.csv


,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,target_organism,target_type,resolution_status
0,EGFR,EGFR,Epidermal growth factor receptor,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN,resolved
1,ERBB2,HER2,Receptor tyrosine-protein kinase erbB-2,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,Homo sapiens,SINGLE PROTEIN,resolved
2,BRAF,BRAF,Serine/threonine-protein kinase B-raf,CHEMBL5145,Serine/threonine-protein kinase B-raf,Homo sapiens,SINGLE PROTEIN,resolved
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,resolved
4,KRAS,KRAS,GTPase KRas,CHEMBL2189121,GTPase KRas,Homo sapiens,SINGLE PROTEIN,resolved
5,VEGFA,VEGFA,Vascular endothelial growth factor A,CHEMBL1783,"Vascular endothelial growth factor A, long form",Homo sapiens,SINGLE PROTEIN,resolved
6,MET,MET,Hepatocyte growth factor receptor,CHEMBL3717,Hepatocyte growth factor receptor,Homo sapiens,SINGLE PROTEIN,resolved
7,PIK3CA,PIK3CA,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",CHEMBL4005,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",Homo sapiens,SINGLE PROTEIN,resolved


## 2. Helper Functions

In [5]:
UNIPROT_API_BASE = "https://rest.uniprot.org/uniprotkb"


def get_json(url, params=None, retries=3, pause=2):
    """GET JSON with simple retries."""
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}; retrying")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}; retrying")
        time.sleep(pause)
    return None


def nested_get(value, path, default=None):
    current = value
    for key in path:
        if isinstance(current, dict):
            current = current.get(key)
        elif isinstance(current, list) and isinstance(key, int) and key < len(current):
            current = current[key]
        else:
            return default
    return default if current is None else current


def get_xrefs(record, database):
    return [
        xref.get("id")
        for xref in record.get("uniProtKBCrossReferences", [])
        if xref.get("database") == database and xref.get("id")
    ]


def join_unique(values, limit=None):
    cleaned = [str(value).strip() for value in values if value is not None and str(value).strip()]
    unique_values = list(dict.fromkeys(cleaned))
    if limit is not None:
        unique_values = unique_values[:limit]
    return " | ".join(unique_values)


def extract_function_summary(record):
    function_texts = []
    for comment in record.get("comments", []) or []:
        if comment.get("commentType") == "FUNCTION":
            for text_obj in comment.get("texts", []) or []:
                text = text_obj.get("value")
                if text:
                    function_texts.append(text)
    return " ".join(function_texts)


def extract_subcellular_locations(record):
    locations = []
    for comment in record.get("comments", []) or []:
        if comment.get("commentType") == "SUBCELLULAR LOCATION":
            for item in comment.get("subcellularLocations", []) or []:
                location = nested_get(item, ["location", "value"])
                if location:
                    locations.append(location)
    return join_unique(locations)


def extract_disease_notes(record):
    notes = []
    for comment in record.get("comments", []) or []:
        if comment.get("commentType") == "DISEASE":
            disease_name = nested_get(comment, ["disease", "diseaseId"])
            disease_desc = nested_get(comment, ["disease", "description"])
            if disease_name and disease_desc:
                notes.append(f"{disease_name}: {disease_desc}")
            elif disease_name:
                notes.append(disease_name)
    return join_unique(notes, limit=5)

## 3. Download UniProt Records

Each raw record is saved so we can reproduce the processed target metadata later.

In [6]:
uniprot_raw = {}

for target_symbol, accession in TARGET_UNIPROT_ACCESSIONS.items():
    url = f"{UNIPROT_API_BASE}/{accession}.json"
    print(f"Fetching {target_symbol}: {accession}")
    record = get_json(url)

    if record is None:
        print(f"  failed: {target_symbol}")
        uniprot_raw[target_symbol] = {"accession": accession, "record": None, "status": "failed"}
    else:
        uniprot_raw[target_symbol] = {"accession": accession, "record": record, "status": "working"}

    time.sleep(0.25)

raw_file = RAW_DIR / "multi_target_uniprot_raw.json"
with raw_file.open("w") as f:
    json.dump(uniprot_raw, f, indent=2)

print("Saved raw UniProt responses:", raw_file)

Fetching EGFR: P00533
Fetching ERBB2: P04626
Fetching BRAF: P15056
Fetching ALK: Q9UM73
Fetching KRAS: P01116
Fetching VEGFA: P15692
Fetching MET: P08581
Fetching PIK3CA: P42336
Saved raw UniProt responses: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/uniprot/multi_target_uniprot_raw.json


## 4. Build Processed Target Metadata

In [7]:
metadata_rows = []

for target_symbol, payload in uniprot_raw.items():
    record = payload.get("record")
    accession = payload.get("accession")

    if record is None:
        metadata_rows.append({
            "target_symbol": target_symbol,
            "target_display_name": TARGET_DISPLAY_NAMES.get(target_symbol, target_symbol),
            "configured_uniprot_accession": accession,
            "uniprot_accession": accession,
            "source_status": "failed",
            "source": "UniProt",
        })
        continue

    protein_name = nested_get(record, ["proteinDescription", "recommendedName", "fullName", "value"])
    alternative_names = []
    for item in nested_get(record, ["proteinDescription", "alternativeNames"], []) or []:
        alt_name = nested_get(item, ["fullName", "value"])
        if alt_name:
            alternative_names.append(alt_name)

    gene_names = []
    for gene in record.get("genes", []) or []:
        primary = nested_get(gene, ["geneName", "value"])
        if primary:
            gene_names.append(primary)
        for synonym in gene.get("synonyms", []) or []:
            value = synonym.get("value")
            if value:
                gene_names.append(value)

    metadata_rows.append({
        "target_symbol": target_symbol,
        "target_display_name": TARGET_DISPLAY_NAMES.get(target_symbol, target_symbol),
        "configured_uniprot_accession": accession,
        "uniprot_accession": record.get("primaryAccession"),
        "uniprot_id": record.get("uniProtkbId"),
        "protein_name": protein_name,
        "alternative_protein_names": join_unique(alternative_names, limit=8),
        "gene_names": join_unique(gene_names, limit=12),
        "organism": nested_get(record, ["organism", "scientificName"]),
        "taxon_id": nested_get(record, ["organism", "taxonId"]),
        "function_summary": extract_function_summary(record),
        "subcellular_locations": extract_subcellular_locations(record),
        "disease_notes": extract_disease_notes(record),
        "ensembl_ids": join_unique(get_xrefs(record, "Ensembl")),
        "chembl_ids": join_unique(get_xrefs(record, "ChEMBL")),
        "hgnc_ids": join_unique(get_xrefs(record, "HGNC")),
        "pdb_ids": join_unique(get_xrefs(record, "PDB"), limit=20),
        "pdb_count": len(get_xrefs(record, "PDB")),
        "go_ids": join_unique(get_xrefs(record, "GO"), limit=20),
        "go_count": len(get_xrefs(record, "GO")),
        "source_status": payload.get("status", "working"),
        "source": "UniProt",
        "url": f"https://www.uniprot.org/uniprotkb/{record.get('primaryAccession')}/entry",
    })

uniprot_metadata_df = pd.DataFrame(metadata_rows)
print("Metadata rows:", len(uniprot_metadata_df))
display(uniprot_metadata_df)

Metadata rows: 8


,target_symbol,target_display_name,configured_uniprot_accession,uniprot_accession,uniprot_id,protein_name,alternative_protein_names,gene_names,organism,taxon_id,...,ensembl_ids,chembl_ids,hgnc_ids,pdb_ids,pdb_count,go_ids,go_count,source_status,source,url
0,EGFR,EGFR,P00533,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Proto-oncogene c-ErbB-1 | Receptor tyrosine-pr...,EGFR | ERBB | ERBB1 | HER1,Homo sapiens,9606,...,ENST00000275493.7 | ENST00000342916.7 | ENST00...,CHEMBL203,HGNC:3236,1IVO | 1M14 | 1M17 | 1MOX | 1NQL | 1XKK | 1YY9...,354,GO:0009925 | GO:0016323 | GO:0030054 | GO:0009...,83,working,UniProt,https://www.uniprot.org/uniprotkb/P00533/entry
1,ERBB2,HER2 / ERBB2,P04626,P04626,ERBB2_HUMAN,Receptor tyrosine-protein kinase erbB-2,Metastatic lymph node gene 19 protein | Proto-...,ERBB2 | HER2 | MLN19 | NEU | NGL,Homo sapiens,9606,...,ENST00000269571.10 | ENST00000584601.5,CHEMBL1824,HGNC:3430,1MFG | 1MFL | 1MW4 | 1N8Z | 1QR1 | 1S78 | 2A91...,59,GO:0016324 | GO:0009925 | GO:0016323 | GO:0005...,65,working,UniProt,https://www.uniprot.org/uniprotkb/P04626/entry
2,BRAF,BRAF,P15056,P15056,BRAF_HUMAN,Serine/threonine-protein kinase B-raf,Proto-oncogene B-Raf | p94 | v-Raf murine sarc...,BRAF | BRAF1 | RAFB1,Homo sapiens,9606,...,ENST00000646891.2,CHEMBL5145,HGNC:1097,1UWH | 1UWJ | 2FB8 | 2L05 | 3C4C | 3D4Q | 3IDP...,131,GO:0044297 | GO:0005737 | GO:0005829 | GO:0098...,31,working,UniProt,https://www.uniprot.org/uniprotkb/P15056/entry
3,ALK,ALK,Q9UM73,Q9UM73,ALK_HUMAN,ALK tyrosine kinase receptor,Anaplastic lymphoma kinase,ALK,Homo sapiens,9606,...,ENST00000389048.8,CHEMBL4247,HGNC:427,2KUP | 2KUQ | 2XB7 | 2XBA | 2XP2 | 2YFX | 2YHV...,79,GO:0070062 | GO:0005886 | GO:0032991 | GO:0043...,22,working,UniProt,https://www.uniprot.org/uniprotkb/Q9UM73/entry
4,KRAS,KRAS,P01116,P01116,RASK_HUMAN,GTPase KRas,K-Ras 2 | Ki-Ras | c-K-ras | c-Ki-ras,KRAS | KRAS2 | RASK2,Homo sapiens,9606,...,ENST00000256078.10 | ENST00000311936.8 | ENST0...,CHEMBL2189121,HGNC:6407,1D8D | 1D8E | 1KZO | 1KZP | 1N4P | 1N4Q | 1N4R...,427,GO:0005737 | GO:0009898 | GO:0005829 | GO:0005...,33,working,UniProt,https://www.uniprot.org/uniprotkb/P01116/entry
5,VEGFA,VEGFA,P15692,P15692,VEGFA_HUMAN,"Vascular endothelial growth factor A, long form",Vascular permeability factor,VEGFA | VEGF,Homo sapiens,9606,...,ENST00000324450.11 | ENST00000372055.9 | ENST0...,CHEMBL1783,HGNC:12680,1BJ1 | 1CZ8 | 1FLT | 1KAT | 1KMX | 1MJV | 1MKG...,56,GO:0005912 | GO:0009986 | GO:0005737 | GO:0005...,127,working,UniProt,https://www.uniprot.org/uniprotkb/P15692/entry
6,MET,MET,P08581,P08581,MET_HUMAN,Hepatocyte growth factor receptor,HGF/SF receptor | Proto-oncogene c-Met | Scatt...,MET,Homo sapiens,9606,...,ENST00000318493.11 | ENST00000397752.8 | ENST0...,CHEMBL3717,HGNC:7029,1FYR | 1R0P | 1R1W | 1SHY | 1SSL | 2G15 | 2RFN...,120,GO:0009925 | GO:0009986 | GO:0005576 | GO:0016...,29,working,UniProt,https://www.uniprot.org/uniprotkb/P08581/entry
7,PIK3CA,PIK3CA,P42336,P42336,PK3CA_HUMAN,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...","Phosphatidylinositol 4,5-bisphosphate 3-kinase...",PIK3CA,Homo sapiens,9606,...,ENST00000263967.4 | ENST00000876545.1 | ENST00...,CHEMBL4005,HGNC:8975,2ENQ | 2RD0 | 3HHM | 3HIZ | 3ZIM | 4JPS | 4L1B...,120,GO:0005737 | GO:0005829 | GO:0014704 | GO:0030...,57,working,UniProt,https://www.uniprot.org/uniprotkb/P42336/entry


In [8]:
metadata_file = PROCESSED_DIR / "multi_target_uniprot_target_metadata.csv"
uniprot_metadata_df.to_csv(metadata_file, index=False)

print("Saved:", metadata_file)
print("Rows:", len(uniprot_metadata_df))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_uniprot_target_metadata.csv
Rows: 8


## 5. Coverage Summary

This shows whether UniProt metadata was successfully collected for each target.

In [9]:
coverage_df = uniprot_metadata_df[[
    "target_symbol", "target_display_name", "configured_uniprot_accession", "uniprot_accession",
    "uniprot_id", "protein_name", "organism", "ensembl_ids", "chembl_ids", "hgnc_ids",
    "pdb_count", "go_count", "source_status",
]].copy()

coverage_df["has_uniprot_record"] = coverage_df["source_status"].eq("working")
coverage_df["has_function_summary"] = uniprot_metadata_df.get("function_summary", pd.Series(dtype="object")).fillna("").astype(str).str.len() > 0
coverage_df["has_external_ids"] = coverage_df[["ensembl_ids", "chembl_ids", "hgnc_ids"]].fillna("").astype(str).agg("".join, axis=1).str.len() > 0

coverage_file = PROCESSED_DIR / "multi_target_uniprot_coverage_summary.csv"
coverage_df.to_csv(coverage_file, index=False)

print("Saved:", coverage_file)
display(coverage_df)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_uniprot_coverage_summary.csv


,target_symbol,target_display_name,configured_uniprot_accession,uniprot_accession,uniprot_id,protein_name,organism,ensembl_ids,chembl_ids,hgnc_ids,pdb_count,go_count,source_status,has_uniprot_record,has_function_summary,has_external_ids
0,EGFR,EGFR,P00533,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Homo sapiens,ENST00000275493.7 | ENST00000342916.7 | ENST00...,CHEMBL203,HGNC:3236,354,83,working,True,True,True
1,ERBB2,HER2 / ERBB2,P04626,P04626,ERBB2_HUMAN,Receptor tyrosine-protein kinase erbB-2,Homo sapiens,ENST00000269571.10 | ENST00000584601.5,CHEMBL1824,HGNC:3430,59,65,working,True,True,True
2,BRAF,BRAF,P15056,P15056,BRAF_HUMAN,Serine/threonine-protein kinase B-raf,Homo sapiens,ENST00000646891.2,CHEMBL5145,HGNC:1097,131,31,working,True,True,True
3,ALK,ALK,Q9UM73,Q9UM73,ALK_HUMAN,ALK tyrosine kinase receptor,Homo sapiens,ENST00000389048.8,CHEMBL4247,HGNC:427,79,22,working,True,True,True
4,KRAS,KRAS,P01116,P01116,RASK_HUMAN,GTPase KRas,Homo sapiens,ENST00000256078.10 | ENST00000311936.8 | ENST0...,CHEMBL2189121,HGNC:6407,427,33,working,True,True,True
5,VEGFA,VEGFA,P15692,P15692,VEGFA_HUMAN,"Vascular endothelial growth factor A, long form",Homo sapiens,ENST00000324450.11 | ENST00000372055.9 | ENST0...,CHEMBL1783,HGNC:12680,56,127,working,True,True,True
6,MET,MET,P08581,P08581,MET_HUMAN,Hepatocyte growth factor receptor,Homo sapiens,ENST00000318493.11 | ENST00000397752.8 | ENST0...,CHEMBL3717,HGNC:7029,120,29,working,True,True,True
7,PIK3CA,PIK3CA,P42336,P42336,PK3CA_HUMAN,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",Homo sapiens,ENST00000263967.4 | ENST00000876545.1 | ENST00...,CHEMBL4005,HGNC:8975,120,57,working,True,True,True


## 6. Final Check

You should see eight rows if all project targets were processed.

In [10]:
print("UniProt Multi-Target Metadata Complete")
print("=" * 70)
print("Targets configured:", len(TARGET_UNIPROT_ACCESSIONS))
print("Metadata rows:", len(uniprot_metadata_df))
print("Working records:", int(coverage_df["has_uniprot_record"].sum()))
print("Files created:")
print("-", raw_file)
print("-", metadata_file)
print("-", coverage_file)

display(coverage_df)

UniProt Multi-Target Metadata Complete
Targets configured: 8
Metadata rows: 8
Working records: 8
Files created:
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/uniprot/multi_target_uniprot_raw.json
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_uniprot_target_metadata.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_uniprot_coverage_summary.csv


,target_symbol,target_display_name,configured_uniprot_accession,uniprot_accession,uniprot_id,protein_name,organism,ensembl_ids,chembl_ids,hgnc_ids,pdb_count,go_count,source_status,has_uniprot_record,has_function_summary,has_external_ids
0,EGFR,EGFR,P00533,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Homo sapiens,ENST00000275493.7 | ENST00000342916.7 | ENST00...,CHEMBL203,HGNC:3236,354,83,working,True,True,True
1,ERBB2,HER2 / ERBB2,P04626,P04626,ERBB2_HUMAN,Receptor tyrosine-protein kinase erbB-2,Homo sapiens,ENST00000269571.10 | ENST00000584601.5,CHEMBL1824,HGNC:3430,59,65,working,True,True,True
2,BRAF,BRAF,P15056,P15056,BRAF_HUMAN,Serine/threonine-protein kinase B-raf,Homo sapiens,ENST00000646891.2,CHEMBL5145,HGNC:1097,131,31,working,True,True,True
3,ALK,ALK,Q9UM73,Q9UM73,ALK_HUMAN,ALK tyrosine kinase receptor,Homo sapiens,ENST00000389048.8,CHEMBL4247,HGNC:427,79,22,working,True,True,True
4,KRAS,KRAS,P01116,P01116,RASK_HUMAN,GTPase KRas,Homo sapiens,ENST00000256078.10 | ENST00000311936.8 | ENST0...,CHEMBL2189121,HGNC:6407,427,33,working,True,True,True
5,VEGFA,VEGFA,P15692,P15692,VEGFA_HUMAN,"Vascular endothelial growth factor A, long form",Homo sapiens,ENST00000324450.11 | ENST00000372055.9 | ENST0...,CHEMBL1783,HGNC:12680,56,127,working,True,True,True
6,MET,MET,P08581,P08581,MET_HUMAN,Hepatocyte growth factor receptor,Homo sapiens,ENST00000318493.11 | ENST00000397752.8 | ENST0...,CHEMBL3717,HGNC:7029,120,29,working,True,True,True
7,PIK3CA,PIK3CA,P42336,P42336,PK3CA_HUMAN,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",Homo sapiens,ENST00000263967.4 | ENST00000876545.1 | ENST00...,CHEMBL4005,HGNC:8975,120,57,working,True,True,True
